# Chapter 9: Recompute paired uncertainty and inspect coverage

Chapter 9 turns discoveries into findings through frozen rules, independent confirmation and uncertainty stated for the declared family of claims. This notebook recomputes the fresh run's paired intervals from the confirmation observations and summarizes the live answer outcomes. Recompute the predeclared paired differences using the same confirmation observations for both candidates. Separately retain all live-agent outcome categories; missing or malformed outputs are not correct answers.

**Evidence.** Run 20260923T220909Z repeated the regional study end to end on two machines. `spark-fe12` ran numerical discovery with three new optimizer seeds (74101 to 74103), one live structural proposal and a fresh 200,000-observation confirmation sample. `spark-ef84` ran 70 live answer tests and nine report drafts with `claude-sonnet-5`. The population and frozen logistic predictor are those of the chapter companions; the seeds, proposal and confirmation sample are new, so these numbers should not be combined with the historical run's.

## Setup and source checks

The setup cell locates the data root, the directory whose relative layout the evidence records use. It uses `MODEL_VALIDATION_ROOT` when that variable is set. Otherwise it searches for a repository checkout containing the current directory and then for the copy installed with the `aiv` package. `R` is this run's folder under the data root, and `read` loads a JSON file from it.

`reference.json` was exported from the run's regional results and records their SHA-256 digest. The final assertion recomputes that digest, so every value used below comes from the unchanged results of this run. This notebook reads recorded outputs; rerunning it makes no language-model call.

In [1]:
from pathlib import Path
import json, hashlib, os, platform
import numpy as np
import pandas as pd

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
LIVE = ROOT / "book/experiments/regional_live"
R = ROOT / "book/experiments/regional_live/runs/20260923T220909Z"
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
print("Run:", R.relative_to(ROOT).as_posix())
read=lambda p: json.loads((R/p).read_text(encoding="utf-8"))
results=read('regional/results.json')
reference=read('reference.json')
seeds=reference['seeds']
assert hashlib.sha256((R/'regional/results.json').read_bytes()).hexdigest()==reference['source_sha256']


Python: 3.12.3
Data root found from: repository checkout
Run: book/experiments/regional_live/runs/20260923T220909Z


## Recompute the paired intervals

For each prespecified pair of arms at a seed, the difference in confirmation contrast is estimated on the same observations. Its standard error uses the difference of the two regions' influence functions,

$$\psi_i=\frac{m_i(L_i-\bar L_m)}{\pi}-\frac{(1-m_i)(L_i-\bar L_{\neg m})}{1-\pi},$$

which removes the variation the regions share. The Bonferroni critical value $z=\Phi^{-1}(1-0.05/38)$ covers the study's 19 claims. The cell asserts that each recomputed interval matches the saved one, then displays the comparisons, the live answer verdicts by phase and the recorded scope statement.

In [2]:
from scipy.stats import norm
a=np.load(R/'regional/confirmation.npz');loss=(a['y']-a['p'])**2
z=norm.ppf(1-.05/(2*19))
def influence(key):
 m=a[key];p=m.mean()
 return m*(loss-loss[m].mean())/p-(~m)*(loss-loss[~m].mean())/(1-p)
for row in results['comparisons']:
 ka=f"{row['a']}_{row['seed']}";kb=f"{row['b']}_{row['seed']}"
 diff=results['confirmation'][ka]['contrast']-results['confirmation'][kb]['contrast']
 se=(influence(ka)-influence(kb)).std(ddof=1)/np.sqrt(len(loss))
 assert np.allclose([diff-z*se,diff+z*se],row['simultaneous_ci'])
display(pd.DataFrame(results['comparisons']))
rows=pd.DataFrame(read('agent_results.json')['rows'])
display(rows.groupby(['phase','verdict']).size())
print(results['scope'])

,seed,a,b,contrast_difference,standard_error,simultaneous_ci
0,74101,reflection,continuation,0.012834,0.001590,"[0.008051940751484483, 0.017616414017858083]"
1,74101,twinning,random,0.000000,0.000000,"[0.0, 0.0]"
2,74102,reflection,continuation,-0.001141,0.001139,"[-0.0045669455524634285, 0.0022844287437013173]"
3,74102,twinning,random,0.000000,0.000000,"[0.0, 0.0]"
4,74103,reflection,continuation,-0.000902,0.001287,"[-0.004772999696062933, 0.002968597897497625]"
5,74103,twinning,random,0.006413,0.001018,"[0.003349358407588642, 0.009475817055983117]"


phase      verdict           
discovery  correct_abstention    36
           correct_answer        28
reserved   correct_abstention     3
           correct_answer         3
dtype: int64

Three optimizer seeds share one discovery sample, one reflection and one confirmation sample; not independent study replications.


**Read the output.** All six intervals are reproduced. The derived-feature revision is ahead of continuation at seed 74101 (0.012834, interval [0.008052, 0.017616]) and unresolved at the other two seeds. Twinning is ahead of random at seed 74103 (0.006413, interval [0.003349, 0.009476]); at the other seeds the two arms froze the same region and their difference is zero. The answer verdicts total 36 correct abstentions and 28 correct answers on discovery and three of each on the reserved tasks, with no wrong, unsupported or unscorable outcome. The printed scope applies throughout: the seeds share one discovery sample, one proposal and one confirmation sample.